# T4 — DVC: versionamento de dados

**Objetivo:** versionar dados como versionamos código, sem colocar os dados no Git, e ligar cada modelo
à versão exata dos dados que o treinou.

| Peça | Onde fica | Vai para o Git? |
|---|---|---|
| `bike.csv` (o dado) | sua pasta + cache `.dvc/cache` + **remote** | ❌ |
| `bike.csv.dvc` (ponteiro: hash md5 + tamanho) | ao lado do dado | ✅ |
| `.dvc/config` (onde fica o remote) | `.dvc/` | ✅ |

> 💡 O DVC e o remote (`../dvc-storage`) foram configurados na T1. Lá, o pipeline só **anotava o hash** do
> `bike.csv` no `dvc.lock`. Agora o próprio dado passa a ser versionado.
>
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

In [2]:
%%bash
git status --short
git log --oneline | head -5
dvc remote list

 M aula01/t4_dvc_dados.ipynb
?? mlflow.log


17f8298 train.py: git_dirty considera só código, parâmetros, pipeline e dados
c283844 Dados v1: 2

011
aab93b7 Merge do repositório original (upstream): traz a T4 e as seções 'Para discutir'
5226e

ca Add files via upload
9efcf64 T3: versões com descrição e tags de origem, executa o notebook e 

registra conclusão


local   /workspaces/dvc-storage (default)


## 1. Versionar os dados v1 (2011)

São sempre os mesmos 4 passos. **Guarde esta receita: você vai repeti-la na Parte 2.**

| Passo | Comando | O que faz |
|---|---|---|
| 1 | `dvc add data/raw/bike.csv` | calcula o md5, guarda no cache e cria o ponteiro `.dvc` |
| 2 | `git add data/raw/bike.csv.dvc` + `git commit` | o **ponteiro** vai para o Git |
| 3 | `git tag dados-v1` | um nome fácil para esta versão |
| 4 | `dvc push` | o **dado** vai para o remote |

In [3]:
%%bash
dvc add data/raw/bike.csv
cat data/raw/bike.csv.dvc

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc



To enable auto staging, run:

	dvc config core.autostage true


outs:
- md5: fc7d423afbce5699a61ac1a8acc533b8
  size: 562742
  hash: md5
  path: bike.csv


O `.gitignore` do template já ignora CSVs dentro de `data/`, por isso o DVC não precisou criar um novo.

In [4]:
%%bash
git add data/raw/bike.csv.dvc
git commit -m "Dados v1: 2011"
git tag dados-v1
dvc push

On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your

 local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be co

mmitted)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   aula01

/t4_dvc_dados.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committe

d)
	mlflow.log

no changes added to commit (use "git add" and/or "git commit -a")


fatal: tag 'dados-v1' already exists


Everything is up to date.

## 2. Desastre: apagaram o dado **e** o cache

Se o dado está no remote, basta um `dvc pull`.

In [5]:
%%bash
# rm -rf data/raw/bike.csv .dvc/cache
dvc pull
wc -l data/raw/bike.csv

Everything is up to date.


8646 data/raw/bike.csv


## 3. Como voltar no tempo (você vai usar na Parte 2)

Para trazer uma versão antiga são **2 comandos**: o Git traz o **ponteiro** antigo e o
`dvc checkout` traz o **dado** correspondente.

```bash
git checkout <tag> -- data/raw/bike.csv.dvc   # ponteiro da versão <tag>
dvc checkout                                  # dado que corresponde ao ponteiro
```
Para voltar à versão atual, troque `<tag>` por `HEAD`.

---
# Parte 2 · Sozinhos

Chegaram dados novos: o **1º semestre de 2012**. Versione-os com a mesma receita da Parte 1.
Repare no `dvc status`: ele mostra que o dado mudou **e** que o estágio `prepare` do pipeline, que depende dele, ficou desatualizado.

In [6]:
%%bash
python src/get_data.py --until 2012-06-30
dvc status

data/raw/bike.csv: 13,003 linhas | 2011-01-01 a 2012-06-30


prepare:


	changed deps:


		modified:           data/raw/bike.csv
train:
	changed deps:
		modified:           src/train.py
dat

a/raw/bike.csv.dvc:
	changed outs:
		modified:           data/raw/bike.csv


### 1. Versione os dados v2 (receita do passo 1)

In [7]:
%%bash
dvc add data/raw/bike.csv
git add data/raw/bike.csv.dvc
git commit -m "Dados v2: até jun/2012"
git tag dados-v2
dvc push
git log --oneline --decorate | head -5

⠋

 Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc



To enable auto staging, run:

	dvc config core.autostage true


[main 5f56dfb] Dados v2: até jun/2012
 1 file changed, 2 insertions(+), 2 deletions(-)


1 file pushed


5f56dfb (HEAD -> main, tag: dados-v2) Dados v2: até jun/2012
17f8298 train.py: git_dirty considera 

só código, parâmetros, pipeline e dados
c283844 (tag: dados-v1) Dados v1: 2011
aab93b7 (origin/ma

in, origin/HEAD) Merge do repositório original (upstream): traz a T4 e as seções 'Para discutir'


5226eca Add files via upload


### 2. Viagem no tempo: volte para a v1, conte as linhas e retorne para a v2 (passo 3)

In [8]:
%%bash
git checkout dados-v1 -- data/raw/bike.csv.dvc   # ponteiro da v1
dvc checkout                                     # dado da v1
wc -l data/raw/bike.csv      # v1: 8.646 linhas (8.645 + cabeçalho)

M       data/raw/bike.csv


8646 data/raw/bike.csv


In [9]:
%%bash
git checkout HEAD -- data/raw/bike.csv.dvc       # ponteiro atual (v2)
dvc checkout                                     # dado da v2
wc -l data/raw/bike.csv      # v2: 13.004 linhas

M       data/raw/bike.csv


13004 data/raw/bike.csv


### 3. Retreine o pipeline com os dados v2 (o `dvc repro` da T1)

Como o dado mudou, o pipeline inteiro roda de novo. O treino gera um novo run no MLflow e uma nova versão no Registry.

In [10]:
%%bash
dvc repro
dvc metrics show
# Comparação com a v1 (atenção: o período de teste mudou, ver a conclusão)
dvc metrics diff dados-v1

'data/raw/bike.csv.dvc' didn't change, skipping


Running stage 'prepare':


> python src/prepare.py


prepare: 13,003 linhas (0 removidas) e 14 colunas -> data/prepared/bike.csv


Updating lock file 'dvc.lock'


Running stage 'split':


... (31 outputs left out)

35.93098     1.99957
metrics.json      r2        0.76221     0.94716      0.18495
metrics.json      

rmse      50.71527    51.02841     0.31314
metrics_val.json  mae       33.38664    38.80813     5.42

148
metrics_val.json  r2        0.79727     0.92702      0.12975
metrics_val.json  rmse      54.3423

3    58.41901     4.07668


### 4. Guarde o recibo e publique no GitHub: o código, os ponteiros e as tags (os dados não vão)

In [11]:
%%bash
git add dvc.lock metrics.json metrics_val.json reports/previsto_vs_real.png
git commit -m "Pipeline retreinado com os dados v2"
dvc push
# git push
# git push --tags
git log --oneline --decorate | head -5

[main 15bb99d] Pipeline retreinado com os dados v2
 4 files changed, 33 insertions(+), 33 deletions(

-)


8 files pushed


15bb99d (HEAD -> main) Pipeline retreinado com os dados v2
5f56dfb (tag: dados-v2) Dados v2: até ju

n/2012
17f8298 train.py: git_dirty considera só código, parâmetros, pipeline e dados
c283844 (tag

: dados-v1) Dados v1: 2011
aab93b7 (origin/main, origin/HEAD) Merge do repositório original (upstre

am): traz a T4 e as seções 'Para discutir'
